In [ ]:
from collections import OrderedDict
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import torch
import torch.nn.functional as F
from torch import Tensor, nn
import torchvision
from torchvision.models import DenseNet121_Weights, densenet121
from torchvision import transforms
from torch.utils.data import DataLoader

from xdl_densecaps.datasets.binary_image_dataset import BinaryNormalLesionDataset
from xdl_densecaps.models.backbone import DenseNet121FeatureExtractor


## HYPERPARAMS

In [ ]:
TEST_DATA_DIR = "data\\raw"
MODEL_PATH = "artifacts/normal_lesion_densenet121/best.pt"

# BINARY_THRESHOLD = 0.022
IMG_SIZE = 128
BATCH_SIZE = 8
BINARY_THRESHOLD = 0.5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Stage 1: DenseNet + Capsnet Hybrid

image → DenseNet first 4 blocks → CapsNet head → margin loss

## DenseNet Backbone feature extractor

In [ ]:
backbone_features = DenseNet121FeatureExtractor(
    pretrained=False,
    checkpoint_path=MODEL_PATH,
).to(DEVICE)

print(f"Backbone load result: {backbone_features.load_result}")

In [ ]:
# DenseNet121FeatureExtractor contains the truncated DenseNet feature layers.


In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

dataset = BinaryNormalLesionDataset(root_dir=TEST_DATA_DIR, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
test_batch = next(iter(dataloader))
print(f"Batch shape: {test_batch[0].shape}")
print(f"Class counts: {dataset.class_counts()}")
print(f"Batch labels: {test_batch[1].tolist()}")

In [ ]:
raw = test_batch[0]
data_input = test_batch[0].to(DEVICE)
data_label = test_batch[1].to(DEVICE)

print(f"Input: {data_input.shape}")
with torch.no_grad():
    data = backbone_features(data_input)
print(f"DenseNet features: {data.shape}")

data_extracted = data

## Capsnet

In [ ]:
from xdl_densecaps.models.densenet_capsnet import DenseNetCapsHead, CapsuleMarginLoss


In [ ]:
# Capsule model classes are imported from src/xdl_densecaps/models/densenet_capsnet.py.


In [ ]:
caps_head = DenseNetCapsHead(
    in_channels=1024,
    feature_h=4,
    feature_w=4,
    num_classes=2,
    num_capsules=256,
    capsule_routing_iters=2,
    digit_routing_iters=3,
).to(DEVICE)

criterion = CapsuleMarginLoss(lambda_=0.5).to(DEVICE)

# expected: (BATCH_SIZE, 1024, 4, 4)

probs, digit_caps = caps_head(data)

loss = criterion(probs, data_label)
loss.backward()

# Lesion filter

First, we do global average pooling

In [ ]:
# data_pooled = F.adaptive_avg_pool2d(data, data.shape[2:])
data_pooled = torch.mean(data, dim = 1)
data_pooled.shape

Then, we binarize it.

In [ ]:
import cv2
_, data_binary = cv2.threshold(data_pooled.cpu().detach().numpy(), BINARY_THRESHOLD, 1, cv2.THRESH_BINARY)
# remember, i use binary_inv because the results that i want for the mask is inverted maybe because im using untrained weights rn.
data_binary = torch.from_numpy(data_binary).to(data.device)
data_binary.shape

Next, we can first visuualize the raw image, pooled, and binary data side by side to see the difference.

In [ ]:
index = 0
img_binary = data_binary[index]
img_pooled = data_pooled[index]
raw_img = raw[index]
# Transpose the raw image to (224, 224, 3) for visualization
raw_img = raw_img.permute(1, 2, 0)
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].imshow(raw_img.cpu().detach().numpy())
axs[0].set_title("Raw Image")
axs[1].imshow(img_pooled.cpu().detach().numpy(), cmap='viridis')
axs[1].set_title("Pooled Data")
axs[2].imshow(img_binary.cpu().detach().numpy(), cmap='gray')
axs[2].set_title("Binary Data")

# set colorbar for pooled and binary data (make colorbar only range from 0 to 1)
fig.colorbar(axs[1].imshow(img_pooled.cpu().detach().numpy(), cmap='viridis'), ax=axs[1])
fig.colorbar(axs[2].imshow(img_binary.cpu().detach().numpy(), cmap='gray', vmin=0, vmax=1), ax=axs[2])
plt.show()

Next is check connectivity and aspect ratio

In [ ]:
def _four_connected_neighbors(mask: torch.Tensor) -> torch.Tensor:
    neighbors = torch.zeros_like(mask, dtype=torch.bool)

    neighbors[1:, :] |= mask[:-1, :]
    neighbors[:-1, :] |= mask[1:, :]
    neighbors[:, 1:] |= mask[:, :-1]
    neighbors[:, :-1] |= mask[:, 1:]

    return neighbors


def _square_bbox_mask(
    binary_mask: torch.Tensor,
    y_min: torch.Tensor,
    x_min: torch.Tensor,
    y_max: torch.Tensor,
    x_max: torch.Tensor,
) -> torch.Tensor:
    height, width = binary_mask.shape
    device = binary_mask.device

    bbox_height = y_max - y_min + 1
    bbox_width = x_max - x_min + 1
    side = torch.maximum(bbox_height, bbox_width)

    max_side = torch.tensor(min(height, width), device=device, dtype=side.dtype)
    side = torch.minimum(side, max_side)

    y_start = y_min - (side - bbox_height) // 2
    x_start = x_min - (side - bbox_width) // 2

    zero = torch.zeros((), device=device, dtype=side.dtype)
    max_y_start = torch.tensor(height, device=device, dtype=side.dtype) - side
    max_x_start = torch.tensor(width, device=device, dtype=side.dtype) - side

    y_start = torch.minimum(torch.maximum(y_start, zero), max_y_start)
    x_start = torch.minimum(torch.maximum(x_start, zero), max_x_start)
    y_end = y_start + side - 1
    x_end = x_start + side - 1

    ys = torch.arange(height, device=device).view(-1, 1)
    xs = torch.arange(width, device=device).view(1, -1)
    return (ys >= y_start) & (ys <= y_end) & (xs >= x_start) & (xs <= x_end)


def separate_connected_components_as_square_boxes(
    binary_mask: torch.Tensor,
    aspect_ratio_threshold: float = 4.0,
    min_area: int = 2,
) -> torch.Tensor:
    """
    Returns square bool masks around each connected component.

    Args:
        binary_mask: 2D tensor [H, W].
        aspect_ratio_threshold: keeps component bboxes whose width / height is between
            1 / threshold and threshold.
        min_area: minimum component pixel count before creating a square box.

    Returns:
        Bool tensor [num_square_boxes, H, W].
    """
    if binary_mask.dim() != 2:
        raise ValueError(f"Expected a 2D mask [H, W], got shape {tuple(binary_mask.shape)}")

    binary_mask = binary_mask.bool()
    unvisited = binary_mask.clone()
    square_boxes = []

    while unvisited.any():
        seed_index = torch.nonzero(unvisited, as_tuple=False)[0]
        y, x = seed_index[0], seed_index[1]

        component = torch.zeros_like(binary_mask, dtype=torch.bool)
        frontier = torch.zeros_like(binary_mask, dtype=torch.bool)

        component[y, x] = True
        frontier[y, x] = True
        unvisited[y, x] = False

        while frontier.any():
            neighbors = _four_connected_neighbors(frontier)
            new_frontier = neighbors & unvisited

            if not new_frontier.any():
                break

            component |= new_frontier
            unvisited &= ~new_frontier
            frontier = new_frontier

        area = component.sum()
        if area < min_area:
            continue

        coords = torch.nonzero(component, as_tuple=False)
        y_min, x_min = coords.min(dim=0).values
        y_max, x_max = coords.max(dim=0).values

        width = (x_max - x_min + 1).float()
        height = (y_max - y_min + 1).float()
        aspect_ratio = width / height.clamp_min(1.0)

        if (
            aspect_ratio < aspect_ratio_threshold
            and aspect_ratio > 1.0 / aspect_ratio_threshold
        ):
            square_boxes.append(_square_bbox_mask(binary_mask, y_min, x_min, y_max, x_max))

    if not square_boxes:
        return torch.empty(
            (0, *binary_mask.shape),
            dtype=torch.bool,
            device=binary_mask.device,
        )

    return torch.stack(square_boxes, dim=0)


In [ ]:
# def separate_connected_components(
#     binary_mask: torch.Tensor,
#     aspect_ratio_threshold: float = 4.0,
#     min_area: int = 2,
# ) -> torch.Tensor:
#     return separate_connected_components_as_square_boxes(
#         binary_mask,
#         aspect_ratio_threshold=aspect_ratio_threshold,
#         min_area=min_area,
#     )

In [ ]:
def resize_mask_to_input_image(
    square_mask: torch.Tensor,
    image_size: tuple[int, int],
) -> torch.Tensor:
    """Resize a low-resolution [h, w] bool mask to the input image [H, W]."""
    if square_mask.dim() != 2:
        raise ValueError(f"Expected square_mask [h, w], got {tuple(square_mask.shape)}")

    resized_mask = F.interpolate(
        square_mask.bool().float().unsqueeze(0).unsqueeze(0),
        size=image_size,
        mode="nearest",
    ).squeeze(0).squeeze(0)
    return resized_mask.bool()


def extract_zoomed_masked_input_image(
    input_image: torch.Tensor,
    square_mask: torch.Tensor,
    mode: str = "bilinear",
) -> tuple[torch.Tensor, torch.Tensor]:
    """Resize mask to input size, crop the masked region, then zoom it to [C, H, W]."""
    if input_image.dim() != 3:
        raise ValueError(f"Expected input_image [C, H, W], got {tuple(input_image.shape)}")

    image_mask = resize_mask_to_input_image(square_mask, input_image.shape[-2:])
    masked_image = input_image * image_mask.unsqueeze(0).to(dtype=input_image.dtype)

    coords = torch.nonzero(image_mask, as_tuple=False)
    if coords.numel() == 0:
        return torch.zeros_like(input_image), image_mask

    y_min, x_min = coords.min(dim=0).values.tolist()
    y_max, x_max = coords.max(dim=0).values.tolist()
    crop = masked_image[:, y_min : y_max + 1, x_min : x_max + 1]

    interpolate_kwargs = {"size": input_image.shape[-2:], "mode": mode}
    if mode in {"linear", "bilinear", "bicubic", "trilinear"}:
        interpolate_kwargs["align_corners"] = False
    zoomed_image = F.interpolate(crop.unsqueeze(0), **interpolate_kwargs).squeeze(0)
    return zoomed_image, image_mask


lesion_candidates = []
lesion_candidate_masks = []

for i, img in enumerate(data_binary):
    candidate_list = []
    mask_list = []
    les_cand = separate_connected_components_as_square_boxes(img, min_area=2)
    print(f"img idx-{i}: {len(les_cand)} components")
    if len(les_cand) == 0:
        lesion_candidates.append([])
        lesion_candidate_masks.append([])
        continue
    for mask in les_cand:
        masked_image, resized_mask = extract_zoomed_masked_input_image(data_input[i], mask)
        candidate_list.append(masked_image)
        mask_list.append(resized_mask)
    lesion_candidates.append(candidate_list) # shape: [batch][num_les_cand_per_img], each image is [3, 128, 128]
    lesion_candidate_masks.append(mask_list) # shape: [batch][num_les_cand_per_img], each mask is [128, 128]


In [ ]:
for index in range(len(lesion_candidates)):
    if len(lesion_candidates[index]) ==0:
        continue
    for candidate_index in range(len(lesion_candidates[index])):
        raw_img = raw[index]
        lesion_img = lesion_candidates[index][candidate_index]
        lesion_mask = lesion_candidate_masks[index][candidate_index]
        # Transpose the raw image to (224, 224, 3) for visualization
        raw_img = raw_img.permute(1, 2, 0)
        lesion_img = lesion_img.permute(1, 2, 0)

        coords = torch.nonzero(lesion_mask, as_tuple=False)
        y_min, x_min = coords.min(dim=0).values.tolist()
        y_max, x_max = coords.max(dim=0).values.tolist()
        box_width = x_max - x_min + 1
        box_height = y_max - y_min + 1

        fig, axs = plt.subplots(1, 2, figsize=(12, 5))
        axs[0].imshow(raw_img.cpu().detach().numpy())
        axs[0].add_patch(
            Rectangle(
                (x_min, y_min),
                box_width,
                box_height,
                fill=False,
                edgecolor="red",
                linewidth=2,
            )
        )
        axs[0].set_title(f"Raw Image\nIndex: {index} Component: {candidate_index}")
        axs[1].imshow(lesion_img.cpu().detach().numpy())
        axs[1].set_title(f"Lesion Data\nIndex: {index} Component: {candidate_index}")
        plt.show()